# M30 lab solution: Support desk router

**Brief.** Route billing, technical and general questions to separate knowledge bases, escalate negative-sentiment tickets, and report routing accuracy.

The three knowledge bases are the handbook files split by section in `../data/kb/`, and `../data/tickets.jsonl` holds 40 tickets labelled by hand with their route and sentiment. The graph follows lesson 30.1's router with lesson 30.2's sentiment escalation: one structured triage call, then either a human escalation or an answer from the routed knowledge base.

The notebook reads only `../data/` and imports only from `src/`. Run it from this folder; the lab README lists the keys it needs.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, str(Path("../../../../src").resolve()))

from helpers import get_embeddings, get_experientiallabs_llm
from lab_kit import is_hit, load_handbook, load_jsonl

DATA = Path("../data")
llm = get_experientiallabs_llm()
print("LLM:", llm.model_name)

## 1. Three knowledge bases

In [ ]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

ROUTES = ["billing", "technical", "general"]
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
embeddings = get_embeddings()
kbs, sections = {}, {}
for route in ROUTES:
    pages = load_handbook(DATA / "kb" / route)
    docs = [Document(page_content=c, metadata={"source": p["source"]}) for p in pages for c in splitter.split_text(p["text"])]
    kbs[route] = InMemoryVectorStore.from_documents(docs, embeddings).as_retriever(search_kwargs={"k": 3})
    sections[route] = ", ".join(p["title"] for p in pages)
    print(f"{route}: {len(docs)} chunks from {sections[route]}")
tickets = load_jsonl(DATA / "tickets.jsonl")

## 2. Triage, escalation and the answer, as a LangGraph graph

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field


class Triage(BaseModel):
    route: Literal["billing", "technical", "general"]
    sentiment: Literal["negative", "neutral", "positive"] = Field(description="The customer's tone")


TRIAGE_SYSTEM = ("You triage Larkspur Cloud support tickets. Pick the knowledge base that answers the ticket:\n"
                 + "\n".join(f"- {r}: {sections[r]}" for r in ROUTES)
                 + "\nThen rate the customer's tone: negative (angry, frustrated, complaining), neutral, or positive.")
triage = ChatPromptTemplate.from_messages([("system", TRIAGE_SYSTEM), ("human", "{ticket}")]) | llm.with_structured_output(Triage)
answer_chain = ChatPromptTemplate.from_messages([
    ("system", "Answer the support ticket from these handbook passages only, citing the file name in brackets. "
               "If they do not answer it, say a support engineer will follow up."),
    ("human", "Passages:\n{context}\n\nTicket: {ticket}"),
]) | llm | StrOutputParser()


class TicketState(TypedDict):
    ticket: str
    route: str
    sentiment: str
    sources: list[str]
    reply: str


def triage_node(state):
    t = triage.invoke({"ticket": state["ticket"]})
    return {"route": t.route, "sentiment": t.sentiment}


def escalate_node(state):
    return {"reply": f"Escalated to a human agent in the {state['route']} team.", "sources": []}


def answer_node(state):
    docs = kbs[state["route"]].invoke(state["ticket"])
    context = "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)
    return {"reply": answer_chain.invoke({"context": context, "ticket": state["ticket"]}),
            "sources": [d.metadata["source"] for d in docs]}


builder = StateGraph(TicketState)
builder.add_node("triage", triage_node)
builder.add_node("escalate", escalate_node)
builder.add_node("answer", answer_node)
builder.add_edge(START, "triage")
builder.add_conditional_edges("triage", lambda s: "escalate" if s["sentiment"] == "negative" else "answer", ["escalate", "answer"])
builder.add_edge("escalate", END)
builder.add_edge("answer", END)
desk = builder.compile()
print(desk.get_graph().draw_mermaid())

## 3. The 40 tickets

In [ ]:
rows = []
for t in tickets:
    out = desk.invoke({"ticket": t["text"], "route": "", "sentiment": "", "sources": [], "reply": ""})
    rows.append({"id": t["id"], "route": t["route"], "predicted route": out["route"], "sentiment": t["sentiment"],
                 "predicted sentiment": out["sentiment"], "escalated": out["sentiment"] == "negative",
                 "sources": ", ".join(out["sources"]), "reply": out["reply"]})
log = pd.DataFrame(rows)
print(log[["id", "route", "predicted route", "sentiment", "predicted sentiment", "escalated"]].to_string(index=False))

## 4. Results

In [ ]:
correct = log.route == log["predicted route"]
escalated, negative = log.escalated, log.sentiment == "negative"
precision = (escalated & negative).sum() / escalated.sum() if escalated.sum() else float("nan")
recall = (escalated & negative).sum() / negative.sum()
summary = pd.DataFrame([
    {"metric": "routing accuracy", "value": f"{correct.sum()} / {len(log)} ({correct.mean():.2f})"},
    {"metric": "escalation precision", "value": f"{(escalated & negative).sum()} / {escalated.sum()} ({precision:.2f})"},
    {"metric": "escalation recall", "value": f"{(escalated & negative).sum()} / {negative.sum()} ({recall:.2f})"},
])
print(summary.to_string(index=False))
print("\nRouting confusion (rows: label, columns: predicted):")
print(pd.crosstab(log.route, log["predicted route"]).to_string())
print("\nMisrouted tickets:")
for _, r in log[~correct].iterrows():
    print(f"  {r.id}: {r.route} -> {r['predicted route']}: {tickets[int(r.id[1:]) - 1]['text']}")
print("\nThree answered tickets:")
for _, r in log[~log.escalated].head(3).iterrows():
    print(f"  {r.id} [{r.sources}] {r.reply[:240]!r}")

## What this shows

- Routing accuracy and escalation precision are the brief's two numbers. Recall sits beside precision because a router can buy precision by escalating less: a negative ticket it misses goes to the answer node with no human in the loop.
- The confusion table shows which pair of knowledge bases the router mixes up. Tickets that sit on a section boundary (a backup question asked as a plan complaint) are the ones to read; the labels are one author's call, and a disputed label is a reason to merge or redraw two knowledge bases.
- Escalation here is decided by sentiment alone. A neutral ticket the knowledge base cannot answer is still answered; the answer prompt's fallback line is the only guard for it.